## Results

In [ ]:
from pathlib import Path

from statsmodels.stats import proportion
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import numpy as np
import pandas as pd

from scipy.stats import kruskal, chi2_contingency
from scikit_posthocs import posthoc_dunn


In [ ]:
## Load the data and do some basic post-processing.

azimuth_cols = ["Left 45", "Left 30", "Left 15", "Centre", "Right 15", "Right 30", "Right 45"]

raw_results_file = Path.home() / "Documents/MDU/BTH-SPOTIFY STUDIES/Recorded Data/Results_17062025.csv"


tmpheaders = list(map(str, range(16)))
headers = ["time", "testname", "track", "assessor"] + tmpheaders
bth_results = pd.read_csv(raw_results_file, names=headers)
for i in range(1, 15, 2):
    new_col_name = bth_results[str(i)].loc[0]
    new_data = bth_results[str(i+1)]
    bth_results[new_col_name] = new_data
bth_results.drop(tmpheaders, axis=1, inplace=True)
bth_results = bth_results.reindex(columns=["time", "testname", "track", "assessor"] + azimuth_cols)
bth_results["assessor"] = bth_results["assessor"].map(str)


ranks = bth_results[azimuth_cols].rank(axis=1).rename(columns=lambda c: f"{c} rank")
bth_results = bth_results.join(ranks)

def non_centre_prefered(row):
    return prefered_position(row) != "C"

def prefered_position(row):
    max_rank = np.max(row)
    winners = set(np.argwhere(row == max_rank).flatten().tolist())
    if 3 in winners:
        return "C"
    contains_left = bool(winners.intersection({0,1,2}))
    contains_right = bool(winners.intersection({4,5,6}))
    if contains_left and contains_right:
        return 'LR'
    elif contains_left:
        return "L"
    else:
        return "R"



def age_group(age):
    try:
        age = int(age)
        if age < 20:
            return "<20"
        elif 20<=age<30:
            return "20s"
        elif 30<=age<40:
            return "30s"
        else:
            return "40+"
    except:
        return None


def tts_gender(track):
    if track in ["Brown Eyed Girl - Van Morrison", "Fake Plastic Trees - Radiohead", "bad guy - Billie Eilish"]:
        return "female"
    else:
        return "male"

bth_results["non_centre_prefered"] = ranks.apply(non_centre_prefered, axis=1)
bth_results["prefered_position"] = ranks.apply(prefered_position, axis=1)

assessor_info = pd.read_excel(Path.home() / "Documents/MDU/BTH-SPOTIFY STUDIES/Recorded Data/Research_Study_Before_17062025.xlsx")
assessor_info["Experiment ID"] = assessor_info["Experiment ID"].map(str)
assessor_info.set_index("Experiment ID", inplace=True)

bth_results = bth_results.join(assessor_info, on="assessor", how="left")
bth_results["age_range"] = bth_results["Age"].map(age_group)
bth_results["tts_gender"] = bth_results["track"].map(tts_gender)


display(bth_results.head(6))

def describe_ratios(df: pd.DataFrame):
    out = pd.DataFrame(columns=['total', 'count', 'ratio'])
    for col in df:
        total = df[col].count()
        for name, count in df[col].value_counts().items():
            out.loc[f"{col}.{name}"] = [total, count, count/total]
    return out

display(describe_ratios(assessor_info[["Gender", "Music streaming", "Poddradio/Podcast", "Expert"]]))
display(describe_ratios(bth_results[["non_centre_prefered", "prefered_position", "Gender", "Music streaming", "Poddradio/Podcast", "Expert", "age_range", "tts_gender"]]))
display(bth_results.columns)

def _format(x: float | bool | int) -> str:
    if isinstance(x, bool) or isinstance(x, int):
        return str(x)
    elif x < 0.01:
        return f"{x:.2e}"
    else:
        return f"{x:.2f}"

In [ ]:
def make_ratio_table(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(columns=["$S$", "$N$", "$R$"])
    s = df.non_centre_prefered.count()
    n = df.non_centre_prefered.value_counts()[True]
    r = n / s
    out.loc["All"] = [s, n, r]
    for track_name, track_df in df.groupby("track"):
        s = track_df.non_centre_prefered.count()
        n = track_df.non_centre_prefered.value_counts()[True]
        r = n / s
        out.loc[track_name] = [s, n, r]
    out["$S$"] = out["$S$"].astype(int)
    out["$N$"] = out["$S$"].astype(int)
    return out

ratio_table = make_ratio_table(bth_results)
display(ratio_table)
print(ratio_table.to_latex(column_format='lccc', formatters=_format))

In [ ]:
# H0: There are no differences between the ratings given to any of the azimuths
# H1: At least one azimuth differs
# Use ranked data because we can't trust exact values, so use a Kruskal-Wallis test
# Critical value for 95% confidence: 12.59 (there are 7 classes so chi2 distribution with 6 DOF)

z = bth_results[azimuth_cols]
display(z)
data = z.to_numpy().T
res = kruskal(*data)
print(res)

for title, group in bth_results.groupby("track"):
    data = group[azimuth_cols].to_numpy().T
    res = kruskal(*data)
    significant = res.statistic > 12.59
    print(title, significant, res)

# Result: H0 is rejected. At least azimuth is different. Need post-hoc tests to determine which one.

melted = z.reindex().melt(var_name='azimuth', value_name='score')

posthoc_res = posthoc_dunn(melted, val_col='score', group_col='azimuth', p_adjust='bonferroni')[azimuth_cols].loc[azimuth_cols]
display(posthoc_res)

table = pd.DataFrame(columns=posthoc_res.columns)
table.loc["Median"] = z.median().map(str)
table.loc["P-value"] = posthoc_res["Centre"].map(lambda x: f"{x:.3f}")
display(table)
print(table.to_latex(column_format="lccccccc"))


In [ ]:
print(bth_results.columns)

critical_values = {
    # float("nan"),
    1: 3.84,
    2: 5.99,
    3: 7.82,
    4: 9.49,
    5: 11.07,
    6: 12.59,
    28: 41.337,
}

chi2_df = pd.DataFrame(columns=["dof", "chi2-statistic", "critical-value", "p-value", "significant"])

for strata in ["age_range", "Expert", "Gender", "Music streaming", "Poddradio/Podcast", "tts_gender", "track", "assessor"]:
    data = pd.crosstab(bth_results[strata], bth_results["prefered_position"])
    data["NC"] = data[["L", "R", "LR"]].sum(axis=1)
    data.drop(["L", "R", "LR"], axis=1, inplace=True)
    data = data.T
    display(data)
    res = chi2_contingency(data)
    crit = critical_values[res.dof]
    significant = bool(res.statistic > crit)
    chi2_df.loc[strata] = [res.dof, res.statistic, crit, res.pvalue, significant]
    print(res)

display(chi2_df.sort_values(by='p-value'))




print(chi2_df.sort_values(by='p-value').style.format(formatter=_format).to_latex())


In [ ]:
## Mean ranks
from scipy import stats

cols = ["Left 45 rank", "Left 30 rank", "Left 15 rank", "Centre rank", "Right 15 rank", "Right 30 rank", "Right 45 rank"]
track_rank_df = bth_results[["track"] + cols]
x = np.arange(7)
y = track_rank_df[cols].mean()
yerr_ci = stats.t.ppf(0.975, df=len(track_rank_df)-1) * stats.sem(track_rank_df[cols])
yerr_std = track_rank_df[cols].std()

with plt.rc_context({'figure.figsize': (4, 4*0.618), 'font.family': 'Times'}):
    fig, ax = plt.subplots(1,1, layout='constrained')
    plots = ax.boxplot(
        bth_results[cols], 
        vert=True, 
        notch=True, 
        tick_labels=["L45", "L30", "L15", "C", "R15", "R30", "R45"],
    )
    for line in plots['medians']:
        line.set_linestyle(':')
        line.set_color('k')

    ax.set_xlabel("Azimuth")
    ax.set_ylabel("Rank score")
    fig.savefig("rank_boxplots.pdf")

fig